# 7 · Boot the image in disposable virtual machines

## Goal

Verify that two cloned RAW disks boot with different machine IDs, the installer installs to a blank VM disk, and the installed machine boots.

## Prerequisites

Build both images first. QEMU, OVMF and swtpm are provided by the Docker builder. These tests can consume substantial CPU time.

In [ ]:
from pathlib import Path
import subprocess, json
START=Path.cwd().resolve()
ROOT=next((p for p in (START,*START.parents) if (p/"myst.yml").is_file() and (p/"scripts/build.sh").is_file()),None)
assert ROOT is not None, "Run from the SC Linux checkout (make book-lab)"
def run(*argv):
    print("+"," ".join(map(str,argv)),flush=True)
    return subprocess.run([str(a) for a in argv],cwd=ROOT,check=True)
print("Source:",ROOT)

## Action 1 · Check required inputs

In [ ]:
OUTPUT=ROOT/"build/output"
for rel in ("raw/sc-linux.raw","sc-linux.iso"):
    path=OUTPUT/rel
    assert path.is_file() and path.stat().st_size>1024*1024, f"Build missing: {rel}"
    print(rel, f"{path.stat().st_size:,} bytes")
assert (ROOT/"build/builder.id").is_file(), "Builder image ID missing"

## Action 2 · Execute VM acceptance tests

Tests use temporary virtual disks, isolated firmware variables, a software TPM and **no network interface**.

In [ ]:
run("make","smoke")

## Action 3 · Inspect structured boot evidence

In [ ]:
report_file=OUTPUT/"tests/smoke.json"
report=json.loads(report_file.read_text())
assert report.get("status")=="passed", "Smoke report does not indicate success"
assert set(report.get("checks",[]))=={"two-raw-clones","iso-install","installed-boot"}
records=report.get("records",[])
assert len(records)==3 and len({r["machine_id"] for r in records})==3
print("Checks:", ", ".join(report["checks"]))
for index,record in enumerate(records,1):
    print(index,"machine_id=",record["machine_id"],"authorization=",record["authorization"])

## Action 4 · Locate console logs

In [ ]:
for path in sorted((OUTPUT/"tests").glob("*.log")):
    print(path.name, f"{path.stat().st_size:,} bytes")

## Check

Passing QEMU tests demonstrate selected boot and installation paths, but **not** Secure Boot enforcement, authenticated root, remote attestation, live FreeIPA enrollment, or root growth on arbitrary disks.

## Next

[Create a host-specific provisioning seed](08-provision.ipynb).